# 도구를 활용한 Strands 에이전트와 AgentCore 메모리 (장기 메모리)

## 개요
이 노트북은 Strands와 AgentCore 메모리를 사용하여 대화형 AI 에이전트에 장기 메모리 기능을 구현하는 방법을 보여줍니다. 단기 상호작용에서 중요한 정보를 추출하고 통합하여, 에이전트가 시간이 지나도 여러 대화 세션에 걸쳐 핵심 세부사항을 기억할 수 있도록 하는 방법을 배웁니다.

## 튜토리얼 상세 정보
**사용 사례:** 영구 메모리를 갖춘 요리 어시스턴트

| 항목               | 상세 내용                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 장기 메모리 대화형                                                               |
| 에이전트 유형       | 요리 어시스턴트                                                                  |
| 에이전틱 프레임워크 | Strands Agents                                                                   |
| LLM 모델           | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성요소   | AgentCore '사용자 선호도' 메모리 추출, 메모리 저장 및 검색을 위한 메모리 도구       |
| 예제 난이도         | 초급                                                                             |

학습 내용:
- 장기 보존을 위한 추출 전략으로 AgentCore 메모리 구성
- 이전 대화 기록으로 메모리 하이드레이션
- 장기 메모리를 활용하여 대화 세션 간 개인화된 경험 제공
- Strands 에이전트 프레임워크와 AgentCore 메모리 도구 통합

## 시나리오 컨텍스트

이 튜토리얼에서는 고도로 개인화된 레스토랑 추천을 제공하는 요리 어시스턴트 역할을 수행합니다. AgentCore 메모리의 장기 보존과 자동 정보 추출을 활용하여, 에이전트는 여러 대화에 걸쳐 식이 선호도, 좋아하는 요리 등의 사용자 선호도를 기억할 수 있습니다. 이 영구 메모리를 통해 에이전트는 대화가 며칠 또는 몇 주에 걸쳐 진행되더라도 맞춤형 제안과 원활한 사용자 경험을 제공할 수 있습니다. 이 시나리오는 구조화된 메모리 구성과 설정 가능한 전략이 대화형 AI를 단기 기억을 넘어 진정으로 매력적이고 컨텍스트를 인식하는 상호작용으로 어떻게 발전시키는지 보여줍니다.


## 아키텍처

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>


## 사전 요구사항

이 튜토리얼을 실행하려면 다음이 필요합니다:
- Python 3.10+
- Amazon Bedrock AgentCore 메모리 권한이 있는 AWS 자격 증명
- Amazon Bedrock AgentCore SDK
장기 메모리 리소스를 적절한 추출 전략으로 설정하고 환경을 구성하는 것부터 시작하겠습니다!

## 1단계: 환경 설정
필요한 모든 라이브러리를 임포트하고 이 노트북을 작동시키는 데 필요한 클라이언트를 정의하는 것부터 시작하겠습니다.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import time
import logging
import time
from datetime import datetime

Amazon Bedrock 모델과 AgentCore에 대한 적절한 권한이 있는 리전과 역할을 정의합니다.

In [ ]:
import os

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("culinary-memory")

region = os.getenv('AWS_REGION', 'us-west-2')

## 2단계: 장기 메모리 전략으로 메모리 생성

이 섹션에서는 장기 메모리 기능을 갖춘 메모리 리소스를 생성합니다. 이전 단기 메모리 예제와 달리, 이 구현에는 통합된 정보 보존을 가능하게 하는 특정 메모리 전략이 포함됩니다.

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

client = MemoryClient(region_name=region)

memory_name = "CulinaryAssistant"
memory_id = None

In [ ]:
from botocore.exceptions import ClientError

try:
    print("장기 메모리 생성 중...")

    # 장기 메모리 리소스에 더 설명적인 이름 사용
    memory_name = memory_name

    # 사용자 선호도 전략으로 메모리 생성
    memory = client.create_memory_and_wait(
        name=memory_name,
        description="Culinary Assistant Agent with long term memory",
        strategies=[{
                    StrategyType.USER_PREFERENCE.value: {
                        "name": "UserPreferences",
                        "description": "Captures user preferences",
                        "namespaces": ["user/{actorId}/preferences/"]
                    }
                }],
        event_expiry_days=7,
        max_wait=300,
        poll_interval=10
    )

    memory_id = memory['id']
    print(f"메모리가 성공적으로 생성되었습니다. ID: {memory_id}")
    
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # 메모리가 이미 존재하는 경우 해당 ID를 검색
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"메모리가 이미 존재합니다. 기존 메모리 ID 사용: {memory_id}")
except Exception as e:
    # 메모리 생성 중 오류 처리
    logger.info(f"오류: {e}")
    import traceback
    traceback.print_exc()
    # 오류 시 정리 - 부분적으로 생성된 메모리 삭제
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"메모리 정리 완료: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"메모리 정리 실패: {cleanup_error}")

### 장기 메모리 전략 이해

이 메모리 생성의 핵심 차이점은 **메모리 전략**의 추가입니다. 구성 요소를 살펴보겠습니다:

#### 1. 사용자 선호도 메모리 전략

이 전략은 대화에서 사용자 선호도를 자동으로 식별하고 추출합니다:

```python
"userPreferenceMemoryStrategy": {
    "name": "UserPreferences",
    "description": "Captures user preferences",
    "namespaces": ["user/{actorId}/preferences/"]
}
```

#### 2. 메모리 네임스페이스

`namespaces` 파라미터는 추출된 정보가 저장되는 위치를 정의합니다:

```python
"namespaces": ["user/{actorId}/preferences/"]
```

이 메모리 전략은 단순히 대화를 기억하는 것이 아니라, 대화 속 중요한 정보를 실제로 이해하고 체계적으로 정리하여 향후 사용에 활용하는 보다 정교한 메모리 시스템을 만듭니다.

## 3단계: 이전 대화를 메모리에 저장

이 섹션에서는 단기 메모리를 하이드레이션하는 방법을 보여주며, 이는 백그라운드에서 장기 메모리 추출 프로세스를 자동으로 트리거합니다.

### 단기 메모리 하이드레이션

추출 전략이 구성된 메모리 리소스에 대화를 저장하면, 시스템은 추가 코드 없이 자동으로 이 정보를 장기 보존을 위해 처리합니다.

In [ ]:
actor_id = f"user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"foodie-{datetime.now().strftime('%Y%m%d%H%M%S')}"
namespace = f"user/{actor_id}/preferences/"

In [ ]:
previous_messages = [
    ("Hi, I'm John", "USER"),
    ("Hi John, how can I help you with food recommendations today?", "ASSISTANT"),
    ("I'm looking for some vegetarian dishes to try this weekend.", "USER"),
    ("That sounds great! I'd be happy to help with vegetarian recommendations. Do you have any specific ingredients or cuisine types you prefer?", "ASSISTANT"),
    ("Yes, I really like tofu and fresh vegetables in my dishes", "USER"),
    ("Perfect! Tofu and fresh vegetables make for excellent vegetarian meals. I can suggest some stir-fries, Buddha bowls, or tofu curries. Do you have any other preferences?", "ASSISTANT"),
    ("I also really enjoy Italian cuisine. I love pasta dishes and would like them to be vegetarian-friendly.", "USER"),
    ("Excellent! Italian cuisine has wonderful vegetarian options. I can recommend pasta primavera, mushroom risotto, eggplant parmesan, or penne arrabbiata. The combination of Italian flavors with vegetarian ingredients creates delicious meals!", "ASSISTANT"),
    ("I spent 2 hours looking through cookbooks but couldn't find inspiring vegetarian Italian recipes", "USER"),
    ("I'm sorry you had trouble finding inspiring recipes! Let me help you with some creative vegetarian Italian dishes. How about stuffed bell peppers with Italian herbs and rice, spinach and ricotta cannelloni, or a Mediterranean vegetable lasagna?", "ASSISTANT"),
    ("Hey, I appreciate food assistants with good taste", "USER"),
    ("Ha! I definitely try to bring good taste to the table! Speaking of which, shall we explore some more vegetarian Italian recipes that might inspire you?", "ASSISTANT")
]

In [ ]:
print("\n이전 대화로 단기 메모리 하이드레이션 중...")

# 대화 기록을 단기 메모리에 저장
initial = client.create_event(
    memory_id=memory_id,
    actor_id=actor_id,
    session_id=session_id,
    messages=previous_messages,
)
print("대화가 단기 메모리에 저장되었습니다")

대화 메시지가 포함된 이벤트가 올바르게 저장되었는지 확인합니다.

In [ ]:
events = client.list_events(
    memory_id=memory_id,
    actor_id=actor_id,
    session_id=session_id,
    max_results=5
)
events

이 셀은 실행 중 발생하는 상황을 추적하는 데 도움이 되는 정보 메시지를 표시하도록 로깅 시스템을 구성합니다.

### 백그라운드에서 일어나는 일

`create_event` 호출 후 다음이 자동으로 발생합니다:

1. **단기 저장**: 전체 대화가 원시 형태로 저장됩니다
2. **추출 트리거**: 메모리 시스템이 이 메모리에 UserPreference 전략이 구성되어 있음을 감지합니다
3. **백그라운드 처리**: 추가 코드 없이 시스템이 자동으로:
   - 대화에서 선호도 지표를 분석합니다
   - "저는 채식주의자입니다", "이탈리안 요리를 정말 좋아합니다" 같은 발언을 식별합니다
   - 이러한 선호도를 구조화된 데이터로 추출합니다
4. **장기 메모리 통합**: 추출된 선호도가 구성된 네임스페이스 (`user/{actorId}/preferences/`)에 저장됩니다

추출과 통합은 자동으로 이루어집니다 - 에이전트와 대화를 유지하거나 단기 메모리를 하이드레이션하기만 하면 되며, 메모리 생성 시 구성한 전략이 나머지를 처리합니다.

이 자동 프로세스는 단기 대화 기록이 만료된 후에도 중요한 정보가 장기 메모리에 보존되도록 합니다.


## 장기 메모리 검색

이 섹션에서는 장기 메모리에 저장된 추출된 선호도에 접근하는 방법을 살펴봅니다. 대화 턴에 초점을 맞추는 단기 메모리 검색과 달리, 장기 메모리 검색은 추출 및 통합된 구조화된 정보에 접근하는 데 초점을 둡니다.

### 장기 메모리에서 사용자 선호도 접근

장기 메모리에서 정보를 검색하려면 메모리 생성 시 정의한 네임스페이스 구조를 사용합니다:


In [ ]:
# 메모리 추출이 이벤트를 처리할 시간을 확보하기 위해 30초 대기
time.sleep(30)

try:
    # 음식 선호도에 대해 메모리 시스템 조회
    food_preferences = client.retrieve_memories(
        memory_id=memory_id,
        namespace=namespace,
        query="food preferences",
        top_k=3  # 가장 관련성 높은 결과를 최대 3개까지 반환
    )

    if food_preferences:
        print(f"관련 선호도 레코드 {len(food_preferences)}개를 검색했습니다:")
        for i, record in enumerate(food_preferences):
            print(f"\n메모리 {i+1}:")
            print(f"- 내용: {record.get('content', '지정되지 않음')}")
    else:
        print("일치하는 선호도 레코드를 찾지 못했습니다.")

except Exception as e:
    print(f"선호도 레코드 검색 오류: {e}")

이 방법은 필요할 때 관련 메모리를 검색할 수 있게 합니다. 이제 기본기를 배웠으니 에이전트를 구축해 보겠습니다!

## 4단계: 에이전트 생성
이 섹션에서는 네이티브 `agent_core_memory` 도구를 사용하여 AgentCore 메모리를 Strands 에이전트와 통합하는 방법을 살펴봅니다.

#### 장기 메모리 기능을 갖춘 에이전트 설정
메모리가 활성화된 에이전트를 생성하기 위해 Strands 프레임워크를 사용하고 AgentCore 메모리 리소스에 연결합니다.

In [ ]:
from strands import tool, Agent
from strands_tools.agent_core_memory import AgentCoreMemoryToolProvider

In [ ]:
system_prompt = f"""You are the Culinary Assistant, a sophisticated restaurant recommendation assistant.

PURPOSE:
- Help users discover restaurants based on their preferences
- Remember user preferences throughout the conversation
- Provide personalized dining recommendations

You have access to a Memory tool that enables you to:
- Store user preferences (dietary restrictions, favorite cuisines, budget preferences, etc.)
- Retrieve previously stored information to personalize recommendations

"""

In [ ]:
provider = AgentCoreMemoryToolProvider(
    memory_id=memory_id,
    actor_id=actor_id,
    session_id=session_id,
    namespace=namespace
)

agent = Agent(tools=provider.tools, model="global.anthropic.claude-haiku-4-5-20251001-v1:0",system_prompt=system_prompt)

이미 단기 메모리와 장기 메모리를 채워놓았으므로, 에이전트에서 직접 메모리를 검색해 보겠습니다!

In [ ]:
agent("Give me restaurant recommendations in Irvine based on my food preferences")

에이전트가 사용자의 메모리를 검색하기 위해 retrieve_memory_records 메서드를 사용했을 것입니다.

완료되었습니다! 이제 AgentCore 장기 메모리에서 메모리를 검색할 수 있는 작동하는 Strands 에이전트를 갖추었습니다!

## 정리
이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제합니다.

In [ ]:
# 메모리를 삭제하려면 아래 주석을 해제하세요
#client.delete_memory_and_wait(
#        memory_id = memory_id,
#        max_wait = 300,
#        poll_interval =10
#)